In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import precision_score, recall_score, f1_score


In [2]:
def read_conll(file_path):
    sentences = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        words = []
        tags = []

        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words = []
                    tags = []
            else:
                splits = line.split()
                words.append(splits[0])
                tags.append(splits[-1])  # NER/Chunk tag

    return sentences, labels


train_sentences, train_labels = read_conll("train.txt")
val_sentences, val_labels = read_conll("valid.txt")

print(train_sentences[0])
print(train_labels[0])


['-DOCSTART-']
['O']


In [3]:
unique_labels = set(label for doc in train_labels for label in doc)

label_list = list(unique_labels)
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)


['I-ORG', 'O', 'B-ORG', 'I-MISC', 'B-PER', 'I-LOC', 'B-LOC', 'B-MISC', 'I-PER']


In [4]:
train_sentences = train_sentences[:500]
train_labels = train_labels[:500]

val_sentences = val_sentences[:100]
val_labels = val_labels[:100]


In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


In [6]:
def tokenize_and_align(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = torch.tensor(aligned_labels)
    return tokenized_inputs


train_encodings = tokenize_and_align(train_sentences, train_labels)
val_encodings = tokenize_and_align(val_sentences, val_labels)


In [7]:
class NERDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])


train_dataset = NERDataset(train_encodings)
val_dataset = NERDataset(val_encodings)


In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_dir="./logs",
    report_to="none" 
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        temp_preds = []
        temp_labels = []

        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                temp_preds.append(id2label[p_])
                temp_labels.append(id2label[l_])

        true_preds.append(temp_preds)
        true_labels.append(temp_labels)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds)
    }


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


In [12]:
trainer.train()



c:\Users\HP\Documents\INNOMATICS\Gen AI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=0.6063324584960937, metrics={'train_runtime': 284.0475, 'train_samples_per_second': 1.76, 'train_steps_per_second': 0.44, 'total_flos': 16332080832000.0, 'train_loss': 0.6063324584960937, 'epoch': 1.0})

In [13]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs).logits
    predictions = torch.argmax(outputs, dim=2)

    word_ids = inputs.word_ids()
    prev = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != prev:
            print(tokens[word_idx], "->", id2label[predictions[0][idx].item()])
        prev = word_idx


predict("John works at Google in California")


John -> O
works -> O
at -> O
Google -> O
in -> O
California -> B-LOC
